# 05 — Đánh giá tầng UGC và phần chưa rõ

FE03, 2026-09-13. Đọc feature snapshot của notebook 03 và log collector FE02; **không gọi mạng, không sửa DB hoặc classifier**. So sánh các giả thuyết nhận nhạc, tạo cohort discovery dễ duyệt, ghi đầu vào/version/lý do để chạy lại khi có nhãn.

Các số khớp là độ phủ trên dữ liệu đã quan sát, không phải accuracy. Metadata hiện có cho 70 video đã được duyệt (18 cũ + 52 bổ sung); chưa quan sát không có nghĩa non-music. Chỉ 5 nhãn video người dùng xác nhận được điền sẵn. Nhãn kênh dùng làm policy/feature, không dùng làm ground truth cấp video.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, sys
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from notebooks.residual_study import VERSION, hypotheses, evaluate, merge_labels, restore_cohort, render_review
FEATURE_DIR=Path(os.environ.get('AURALYTICA_FE03_FEATURES',str(ROOT/'artifacts/notebook-runs/03_music_features/3ef0ba60cbf2/fe-v1-633cf963')))
LOG=Path(os.environ.get('AURALYTICA_FE03_LOG',str(ROOT/'artifacts/metadata-smoke/fe03-20260913T082334Z/combined70.jsonl')))
FROZEN=Path(os.environ.get('AURALYTICA_FE03_COHORT',str(ROOT/'artifacts/notebook-runs/05_residual/20260913T080143642628Z/cohort_audit.csv')))
PILOT=ROOT/'artifacts/ytmusic-pilot/20260910T173545Z/manifest.json'
HOLDOUT=ROOT/'artifacts/notebook-runs/01_takeout_eda/2461d5f07e3a/signals-v2/random_review.csv'
CHANNELS=ROOT/'artifacts/notebook-runs/channel_labels.csv'
feature_summary=json.loads((FEATURE_DIR/'summary.json').read_text())
source_hash=feature_summary['quality']['source_hash']
frame=pd.read_csv(FEATURE_DIR/'video_features.csv')
assert frame['video_id'].is_unique
for name in ['proxy_seed','content_candidate','shorts_explicit']:
    frame[name]=frame[name].astype(str).str.lower().eq('true')
records=[json.loads(line) for line in LOG.read_text().splitlines() if line.strip()]
runs={r['run']['id']:r['run'] for r in records if r.get('kind')=='run_manifest'}
observations={}
for record in records:
    if record.get('kind')!='metadata_observed': continue
    assert runs[record['run_id']]['source_hash']==source_hash, 'Metadata and feature snapshots differ'
    payload=record['payload'];vid=record['video_id']
    assert vid in set(frame.video_id)
    if vid not in observations or payload['fetched_at']>observations[vid]['payload']['fetched_at']:
        observations[vid]=record
pilot=json.loads(PILOT.read_text())
assert pilot['source_hash']==source_hash
pilot_ids={r['id'] for r in pilot['samples']}
known_music={r['id'] for r in pilot['samples'] if r['stratum']=='user_confirmed_music'}
holdout_ids=set(pd.read_csv(HOLDOUT,usecols=['video_id'])['video_id']) # Never read holdout labels.
channels=pd.read_csv(CHANNELS)
excluded_channels=set(channels.loc[channels['source'].eq('user_confirmed') & channels['label'].isin(['entertainment','podcast','non_music']),'channel_key'])
frame['channel_excluded']=frame['channel_key'].isin(excluded_channels)
frame['player_status']=frame['video_id'].map(lambda v: observations.get(v,{}).get('payload',{}).get('status','not_fetched'))
frame['player_exact_match']=frame['video_id'].map(lambda v: observations.get(v,{}).get('payload',{}).get('exact_match',False))
for name,field in [('player_type','music_video_type'),('category','category'),('playability','playability'),('duration_seconds','duration_seconds')]:
    frame[name]=frame['video_id'].map(lambda v: observations.get(v,{}).get('payload',{}).get('evidence',{}).get(field,{}).get('value'))
frame['observation_id']=frame['video_id'].map(lambda v: observations.get(v,{}).get('id'))
frame['metadata_fetched_at']=frame['video_id'].map(lambda v: observations.get(v,{}).get('payload',{}).get('fetched_at'))
frame['source_hash']=source_hash
frame['manual_label']='';frame.loc[frame.video_id.isin(known_music),'manual_label']='music'
frame['label_source']='';frame.loc[frame.video_id.isin(known_music),'label_source']='user_confirmed_before_pilot'
frame['notes']=''
OUT=ROOT/'artifacts/notebook-runs/05_residual'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True,exist_ok=False)
print('Snapshot videos:',len(frame),'| observed IDs:',len(observations),'| no request:',frame.player_status.eq('not_fetched').sum())
print('Output:',OUT)

## 1. Các giả thuyết cần đối chiếu

So baseline với seed/strong player type, rồi thêm UGC + nội dung, UGC + quay lại nhiều ngày, hoặc UGC + một trong hai. Tất cả nhánh mới giữ các ca có dấu hiệu talk/Shorts hoặc policy kênh ở review. Hashtag chỉ làm thận trọng khi tự chọn, không xác nhận là Shorts.

`repeat_alone_diagnostic` cố tình chỉ dùng số ngày để lộ rủi ro. Category Gaming/Entertainment không chặn nhạc; duration ngắn và UNPLAYABLE không biến thành non-music. Ngưỡng 2/3/5/10 ngày chỉ là sensitivity analysis, **chưa chọn ngưỡng production**.

In [ ]:
masks=hypotheses(frame)
rule_names=['baseline','metadata_strong','ugc_content','ugc_recurrence','ugc_combined','repeat_alone_diagnostic']
coverage=pd.DataFrame({'matched':masks[rule_names].sum(),
    'matched_known_5_music':masks.loc[frame.video_id.isin(known_music),rule_names].sum(),
    'new_vs_baseline':(masks[rule_names] & ~masks['baseline'].to_numpy()[:,None]).sum()})
display(coverage)
sensitivity=[]
for days in [2,3,5,10]:
    candidate=hypotheses(frame,min_days=days)
    sensitivity.append(dict(min_days=days, combined=int(candidate.ugc_combined.sum()),
        known_music=int(candidate.loc[frame.video_id.isin(known_music),'ugc_combined'].sum()),
        repeat_only=int(candidate.repeat_alone_diagnostic.sum()),
        repeat_in_user_excluded_channels=int((candidate.repeat_alone_diagnostic & frame.channel_excluded).sum())))
display(pd.DataFrame(sensitivity))
joined=pd.concat([frame,masks.add_prefix('hypothesis_')],axis=1)
display(joined.loc[frame.video_id.isin(known_music),['title','watch_count','watch_days','player_type','category','content_candidate','hypothesis_metadata_strong','hypothesis_ugc_content','hypothesis_ugc_combined']])
metrics=evaluate(masks[rule_names],frame.manual_label)
print('Prior-label check before importing review CSV:',json.dumps(metrics,ensure_ascii=False))

## 2. Chọn thêm ca dễ nhầm và ca thiếu bằng chứng

Giữ 18 pilot làm regression/discovery, thêm mẫu phân tầng. Mỗi stratum chọn tối đa 2 video/kênh, thứ tự theo hash cố định; ID mới không lấy từ holdout 300. Policy kênh không được dùng để điền nhãn non_music. Podcast/BGM/Shorts dưới đây chỉ là **ứng viên theo văn bản**, phải xem nội dung mới chốt.

Mẫu này không ngẫu nhiên đại diện tổng thể; precision/recall nếu có sau review chỉ mô tả phần có nhãn trong cohort discovery. Ca đã tune không dùng lại làm holdout.

Sau enrichment, giữ cố định cohort đã chọn trước khi gửi 52 ID. Không tự chọn thêm mẫu do các stratum “missing metadata” thay đổi. Nhãn và sample_reason cũ được bảo toàn.

In [ ]:
rest=frame.auto_group.eq('rest')
text=frame.title.fillna('')
short=masks.short_candidate
repeat=pd.to_numeric(frame.watch_days,errors='coerce').ge(3)
strata={
 'music_terms_with_shorts_marker':short & frame.content_candidate,
 'talk_and_music_terms':masks.talk_candidate & frame.content_candidate,
 'repeat_in_excluded_channel':repeat & frame.channel_excluded,
 'repeat_without_content':rest & repeat & ~frame.content_candidate,
 'content_seen_once':rest & frame.content_candidate & frame.watch_count.eq(1),
 'moderate_repeat_2_to_4_days':rest & pd.to_numeric(frame.watch_days,errors='coerce').between(2,4),
 'one_day_repetition':rest & frame.watch_count.ge(3) & pd.to_numeric(frame.watch_days,errors='coerce').eq(1),
 'podcast_token_candidate':text.str.contains(r'podcast|talk\s?show|ポッドキャスト|팟캐스트',case=False,regex=True),
 'speech_candidate':masks.talk_candidate & ~frame.content_candidate,
 'performance_missing_metadata':frame.performance_phrase.eq(True) & frame.player_status.eq('not_fetched'),
 'background_context_candidate':frame.content_candidate & text.str.contains(r'game|gaming|funny|memes|comment|threads|review|reaction',case=False,regex=True),
 'other_residual':rest & frame.player_status.eq('not_fetched'),
}
frozen=pd.read_csv(FROZEN,keep_default_na=False)
cohort=restore_cohort(joined,frozen)
stratum_counts=[]
for name,mask in strata.items():
    stratum_counts.append(dict(stratum=name,total_candidates=int(mask.sum()),
        eligible_outside_holdout=int((mask & ~frame.video_id.isin(holdout_ids)).sum()),
        frozen_sampled=int(cohort.sample_reason.eq(name).sum())))
cohort['url']='https://www.youtube.com/watch?v='+cohort.video_id
cohort['in_old_holdout']=cohort.video_id.isin(holdout_ids)
assert not (set(cohort.video_id)-pilot_ids) & holdout_ids
cohort['eval_role']='discovery_only'
review_path=os.environ.get('AURALYTICA_FE03_LABELS')
if review_path:
    # Archive the exact submitted bytes so later downloads cannot replace this evidence.
    (OUT/'input_review_labels.csv').write_bytes(Path(review_path).read_bytes())
    cohort=merge_labels(cohort,pd.read_csv(OUT/'input_review_labels.csv',keep_default_na=False))
display(pd.DataFrame(stratum_counts))
print('Cohort:',len(cohort),'| labelled:',cohort.manual_label.isin(['music','non_music']).sum())
print('Remove from old holdout after pilot exposure:',int(cohort.in_old_holdout.sum()))

## 3. Bảng review và xuất dữ liệu

Điền `manual_label` là music/non_music/uncertain/unavailable, thêm `notes`. Chỉ gán music khi nội dung chính là âm nhạc; có BGM trong video nói chuyện/game vẫn là non_music. Nhập CSV đã điền qua `AURALYTICA_FE03_LABELS`, chạy lại để có confusion matrix trên các hàng đã resolve. Source hash/ID/nhãn được kiểm tra trước khi ghép.

Mỗi lần chạy xuất folder mới; không ghi đè review cũ. `review.html` chỉ là bảng offline có link mở video; không nhúng ảnh/media và không tự gọi mạng. Cột dự đoán được để trong audit riêng để tránh dùng gợi ý làm nhãn thật.

In [ ]:
review_cols=['video_id','title','channel_name','url','source_hash','manual_label','label_source','notes']
review=cohort[review_cols].copy()
review.to_csv(OUT/'review.csv',index=False)
# Prioritize unresolved contradictions; predictions remain hidden in the review CSV.
unresolved=cohort[~cohort.manual_label.isin(['music','non_music'])]
priority_parts=[];priority_ids=set()
focus=[
 ('type_present_but_guarded',unresolved.player_type.notna() & unresolved.hypothesis_guarded_for_review,4),
 ('selected_outside_music_category',unresolved.hypothesis_ugc_combined & ~unresolved.category.eq('Music'),3),
 ('ugc_recurrence_without_content',unresolved.hypothesis_ugc_recurrence & ~unresolved.hypothesis_ugc_content,2),
 ('official_type_guidance_title',unresolved.player_type.eq('MUSIC_VIDEO_TYPE_OMV') & unresolved.title.str.contains(r'how to|lesson|tutorial',case=False,regex=True),1),
 ('type_present_unplayable',unresolved.player_type.notna() & unresolved.playability.eq('UNPLAYABLE'),2),
]
for reason,mask,cap in focus:
    chosen=unresolved[mask & ~unresolved.video_id.isin(priority_ids)].head(min(cap,12-len(priority_ids))).copy()
    chosen['priority_reason']=reason;priority_parts.append(chosen);priority_ids.update(chosen.video_id)
if len(priority_ids)<12:
    fill=unresolved[~unresolved.video_id.isin(priority_ids)].head(12-len(priority_ids)).copy()
    fill['priority_reason']='other_unresolved';priority_parts.append(fill)
priority=pd.concat(priority_parts,ignore_index=True)
priority[['video_id','priority_reason']].to_csv(OUT/'priority_reasons.csv',index=False)
priority[review_cols].to_csv(OUT/'priority_review.csv',index=False)
request_rows=cohort.loc[cohort.player_status.eq('not_fetched'),['video_id','title','channel_name','sample_reason']]
(OUT/'metadata_request_manifest.json').write_text(json.dumps(dict(source_hash=source_hash, count=len(request_rows),
    status='prepared_not_sent',method='get_song',samples=request_rows.to_dict(orient='records')),ensure_ascii=False,indent=2))
cohort.to_csv(OUT/'cohort_audit.csv',index=False)
joined.to_csv(OUT/'all_video_hypotheses.csv',index=False)
coverage.to_csv(OUT/'coverage.csv')
pd.DataFrame(sensitivity).to_csv(OUT/'sensitivity.csv',index=False)
pd.DataFrame(stratum_counts).to_csv(OUT/'strata.csv',index=False)
cohort.loc[cohort.in_old_holdout,['video_id','eval_role']].to_csv(OUT/'holdout_exclusions.csv',index=False)
ordered_review=pd.concat([priority[review_cols],review[~review.video_id.isin(priority.video_id)]],ignore_index=True)
(OUT/'review.html').write_text(render_review(ordered_review, priority_count=len(priority)),encoding='utf-8')
cohort_predictions=cohort[['hypothesis_'+r for r in rule_names]].copy()
cohort_predictions.columns=rule_names
cohort_metrics=evaluate(cohort_predictions,cohort.manual_label)
# Keep a row-level explanation for every labelled prediction, including correct cases.
labelled=cohort[cohort.manual_label.isin(['music','non_music'])]
audit_parts=[]
for rule in rule_names:
    audit=labelled[['video_id','title','source_hash','manual_label','label_source','notes',
        'watch_count','watch_days','player_type','duration_seconds','observation_id',
        'hypothesis_talk_candidate','hypothesis_short_candidate','channel_excluded']].copy()
    audit['rule']=rule
    audit['predicted_music']=labelled['hypothesis_'+rule]
    audit['outcome']=['tp' if pred and label=='music' else 'fp' if pred else 'fn' if label=='music' else 'tn'
        for pred,label in zip(audit.predicted_music,audit.manual_label)]
    audit_parts.append(audit)
prediction_audit=pd.concat(audit_parts,ignore_index=True)
prediction_audit.to_csv(OUT/'labelled_prediction_audit.csv',index=False)
prediction_audit[prediction_audit.outcome.isin(['fp','fn'])].to_csv(OUT/'classification_errors.csv',index=False)
# Export cumulative labels so the next partial CSV can build on this reviewed cohort.
review.to_csv(OUT/'cumulative_review_labels.csv',index=False)
labelled_sensitivity={str(days):evaluate(hypotheses(cohort,min_days=days)[rule_names],cohort.manual_label)
    for days in [2,3,5,10]}
(OUT/'labelled_sensitivity.json').write_text(json.dumps(labelled_sensitivity,ensure_ascii=False,indent=2))
if review_path:
    submitted_ids=set(pd.read_csv(OUT/'input_review_labels.csv',keep_default_na=False).video_id)
    submitted=cohort.video_id.isin(submitted_ids)
    submitted_metrics=evaluate(cohort_predictions.loc[submitted],cohort.loc[submitted,'manual_label'])
    (OUT/'submitted_labels_evaluation.json').write_text(json.dumps(submitted_metrics,ensure_ascii=False,indent=2))
inputs=[FEATURE_DIR/'video_features.csv',FEATURE_DIR/'summary.json',LOG,PILOT,HOLDOUT,CHANNELS,FROZEN,ROOT/'notebooks/residual_study.py',ROOT/'notebooks/review_template.html',ROOT/'notebooks/05_residual_evaluation.ipynb']
if review_path: inputs.append(OUT/'input_review_labels.csv')
result=dict(version=VERSION,source_hash=source_hash,total_videos=len(frame),metadata_observed=len(observations),
    no_metadata=int(frame.player_status.eq('not_fetched').sum()),cohort_size=len(cohort),
    label_counts=cohort.manual_label.value_counts().to_dict(),
    labelled=int(cohort.manual_label.isin(['music','non_music']).sum()),
    known_music=len(known_music),holdout_exclusions=int(cohort.in_old_holdout.sum()),
    coverage=coverage.astype(int).to_dict(),sensitivity=sensitivity,strata=stratum_counts,
    evaluation=cohort_metrics,inputs_sha256={str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in inputs},
    warning='Discovery only; no calibrated threshold, production decisions or population accuracy.')
(OUT/'summary.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in result.items() if k not in ['inputs_sha256','strata']},ensure_ascii=False,indent=2))
print('Review:',OUT/'review.html')